# ♟️ ChessStyleNet — Colab Training Notebook

Train a style-conditioned chess policy network conditioned on **your own Lichess games** (or rating bands) on a free Colab GPU.

**Pipeline:**
1. Setup environment & GPU
2. Download your Lichess games via the Lichess API (no huge dumps needed!)
3. Train `ChessStyleNet` on your positions & moves
4. Evaluate top-1 / top-3 move prediction accuracy
5. Download checkpoint to use locally with your FastAPI + HTML/JS web app

### Step 1: Environment & GPU Setup
Make sure your runtime is GPU-accelerated: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Clone repository (or upload project files directly)
# If using private repo: !git clone https://<token>@github.com/<username>/chess-style-engine.git
!git clone https://github.com/your-username/chess-style-engine.git
%cd chess-style-engine

!pip install -q -r requirements.txt
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available())"

### Step 2: Download YOUR Games directly from Lichess
Change `LICHESS_USERNAME` below to your own Lichess handle. This downloads your games in seconds and extracts `(fen, move_uci, player_id)` pairs where you made the move.

In [ ]:
import io
import csv
import os
import requests
import chess.pgn

LICHESS_USERNAME = "your_lichess_username"  # <-- CHANGE TO YOUR LICHESS HANDLE
MAX_GAMES = 2000  # Number of games to download

os.makedirs("data", exist_ok=True)
url = f"https://lichess.org/api/games/user/{LICHESS_USERNAME}?max={MAX_GAMES}&perfType=blitz,rapid,classical&moves=true"
print(f"Fetching games for '{LICHESS_USERNAME}' from Lichess API...")

headers = {"Accept": "application/x-chess-pgn"}
resp = requests.get(url, headers=headers, stream=True)
resp.raise_for_status()

pgn_stream = io.StringIO(resp.text)
rows = []
game_count = 0

while True:
    game = chess.pgn.read_game(pgn_stream)
    if game is None:
        break

    white = game.headers.get("White", "").lower()
    black = game.headers.get("Black", "").lower()
    user_lower = LICHESS_USERNAME.lower()
    my_color = chess.WHITE if white == user_lower else (chess.BLACK if black == user_lower else None)

    if my_color is None:
        continue

    board = game.board()
    for move in game.mainline_moves():
        if board.turn == my_color and not board.is_game_over():
            rows.append({
                "fen": board.fen(),
                "move_uci": move.uci(),
                "player_id": LICHESS_USERNAME
            })
        board.push(move)

    game_count += 1
    if game_count % 100 == 0:
        print(f"Processed {game_count} games ({len(rows)} positions)...")

out_file = "data/my_lichess_games.csv"
with open(out_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["fen", "move_uci", "player_id"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Done! Saved {len(rows)} personal positions from {game_count} games to {out_file}")

### Step 3: Run Quick Sanity Unit Tests

In [ ]:
!python -m pytest tests/ -v

### Step 4: Train on GPU
Trains `ChessStyleNet` conditioned on your moves. 15-20 epochs takes only ~2-5 minutes on a Colab T4.

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

!python src/training/train.py \
    --csv data/my_lichess_games.csv \
    --epochs 15 \
    --batch_size 256 \
    --lr 1e-3 \
    --save_path checkpoints/my_style_net.pt

### Step 5: Evaluate Model Accuracy
Check Top-1 and Top-3 accuracy on your moves.

In [ ]:
!python src/evaluation/evaluate.py \
    --ckpt checkpoints/my_style_net.pt \
    --csv data/my_lichess_games.csv

### Step 6: Download Checkpoint to Run Locally
Download `my_style_net.pt` to place into your local `checkpoints/` directory for your FastAPI backend.

In [ ]:
from google.colab import files
files.download("checkpoints/my_style_net.pt")